In [ ]:
import os, sys
os.environ['CUDA_VISIBLE_DEVICES'] = '0,'
from pathlib import Path
import json
from datetime import datetime
from tqdm import tqdm
import numpy as np
from PIL import Image
import torch

from hmr4d.utils.image_gen.nunchaku import ImageGenerator_Nunchaku

device = 'cuda'

In [2]:
img_gen = ImageGenerator_Nunchaku(device=device)
id_prompts = [
    # Male / Unknown
    "A young man with short, slightly wavy hair and light stubble, wearing a fitted beige jacket over a white t-shirt and dark denim jeans, stands confidently, exuding a casual yet expressive gesture. ", 
    # Female / Unknown
    "A young woman with shoulder-length straight hair parted naturally, wearing a fitted neutral-toned blazer over a simple white top and tailored pants, stands upright with a relaxed yet confident posture. ",
    # Male / Middle Eastern
    "A Middle Eastern man in his early thirties with neatly trimmed dark hair and a short beard, dressed in a charcoal wool coat over a dark turtleneck and slim-fit trousers, leans slightly forward with one hand in his pocket, giving a thoughtful, composed presence. ",
    # Female / Black (African descent)
    "A tall Black woman with deep brown skin and natural textured hair styled in a high puff, wearing a modern sleeveless jumpsuit in muted earth tones, stands upright with squared shoulders, projecting strength and elegance. ",
    # Male / White (European descent)
    "A Caucasian man with medium-length tousled blond hair and a clean-shaven face, wearing a relaxed linen shirt with rolled-up sleeves and light chinos, stands casually with a slight hip shift, conveying an effortless, laid-back charm. ", 
    # Female / Latina
    "A Latina woman with shoulder-length wavy dark hair and warm olive skin, dressed in a fitted leather jacket over a soft knit top and high-waisted jeans, stands with crossed arms and a subtle smile, radiating confidence and edge. ",
    # Male / South Asian
    "A South Asian man with neatly combed hair and light stubble, wearing a tailored navy blazer over a crisp white shirt and dark slacks, stands straight with his hands at his sides, presenting a polished and professional demeanor. ",
    # Female / Mixed Ethnicity
    "A mixed-ethnicity woman with short, curly hair dyed deep auburn, wearing an oversized knit sweater and pleated skirt in neutral tones, stands slightly angled with relaxed shoulders, giving an artistic and introspective feel. ",
    # Male / East Asian
    "An East Asian man with straight jet-black hair parted cleanly to the side, wearing a long tailored coat over a monochrome outfit, stands upright with a composed posture and neutral expression, embodying modern urban sophistication. ",
    # Female / East Asian
    "A young East Asian woman with long, straight black hair tied in a low ponytail, wearing a minimalist black blazer over a silk ivory blouse and tailored trousers, stands calmly with her hands loosely clasped, exuding quiet confidence. ", 
]

scene_prompts = [
    # Modern city square
    "The scene takes place in a modern city square paved with smooth stone tiles, surrounded by contemporary glass-and-steel office buildings that reflect the warm afternoon sunlight.", 
    # Indoor architectural lobby
    "The scene takes place inside a spacious modern lobby with polished concrete floors, tall ceilings, and floor-to-ceiling windows that allow natural light to softly fill the space.",
    # Urban park edge
    "The scene is set at the edge of a well-maintained urban park, featuring trimmed greenery, paved walkways, and contemporary buildings visible in the background under clear daylight.",
    # Rooftop terrace
    "The scene takes place on a modern rooftop terrace with glass railings, subtle greenery, and a distant city skyline visible beneath a bright, open sky.",
    # Minimalist indoor studio
    "The scene is set in a minimalist indoor studio with smooth neutral-colored walls, soft diffused lighting, and a clean, uncluttered atmosphere.",
    # Open natural landscape
    "The scene takes place in an open natural landscape with gently rolling terrain, low vegetation, and a broad, unobstructed ground plane, extending toward distant hills beneath a wide, softly lit sky.",
    # Coastal environment
    "The scene is set along a quiet coastline featuring a wide stretch of firm sand, calm ocean waves, and an open horizon, illuminated by soft, even daylight.",
    # Mountain overlook
    "The scene unfolds at a spacious mountain overlook with flat rock surfaces, expansive views of layered ridgelines, and clear natural lighting that emphasizes the open surroundings.",
    # Industrial interior
    "The scene is set inside a converted industrial warehouse with wide open floors, high ceilings, exposed concrete and steel elements, and evenly diffused lighting across the space.",
    # Countryside road
    "The scene takes place along a quiet countryside road with a broad, empty surface and open surroundings, bordered by low vegetation and stretching freely toward the horizon."
]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


[2026-01-27 21:44:14.323] [info] Initializing QuantizedFluxModel on device 0
Injecting quantized module[2026-01-27 21:44:14.385] [info] Loading partial weights from pytorch
[2026-01-27 21:44:16.547] [info] Done.



Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


In [3]:
output_dir = Path("outputs/uni3c_aligned/test_0127")
for output_path in sorted(list(output_dir.glob("*"))):
    openpose_render = Image.open(output_path / "openpose_render.png").convert("RGB")
    width, height = openpose_render.size
    
    prompt = np.random.choice(id_prompts) + np.random.choice(scene_prompts) 
    seed = np.random.randint(2**31)
    coarse_img, coarse_meta = img_gen.step1(
        prompt=prompt,
        image=openpose_render,
        width=width, height=height, 
        generator = torch.Generator(device='cuda').manual_seed(seed)
    )

    final_img, final_meta = img_gen.step2(
        prompt=prompt,
        image=coarse_img[0],
        width=width*2, height=height*2,
        generator = torch.Generator(device='cuda').manual_seed(seed)
    )
    
    coarse_img[0].save(output_path / "reference_coarse.png")
    final_img = final_img[0].resize((width, height))
    final_img.save(output_path / "reference.png")
    (output_path / "img_meta.json").write_text(json.dumps({'coarse_meta': coarse_meta, 'final_meta': final_meta}, indent=4))
    final_img

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/38 [00:00<?, ?it/s]

/home/guangyu/anaconda3/envs/nunchaku/lib/python3.11/site-packages/nunchaku/utils.py:47: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:345.)
  result[[slice(0, extent) for extent in tensor.shape]] = tensor
Token indices sequence length is longer than the specified maximum sequence length for this model (83 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['emphasizes the open surroundings.']


  0%|          | 0/50 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (83 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['emphasizes the open surroundings.']


...

  0%|          | 0/38 [00:00<?, ?it/s]

The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['daylight.']


  0%|          | 0/50 [00:00<?, ?it/s]

The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['daylight.']


  0%|          | 0/38 [00:00<?, ?it/s]